# Pythia 410M prompt playground

This notebook is the smallest practical first step for the semantic-topology project: load `EleutherAI/pythia-410m`, provide a prompt, and inspect the generated continuation.

Pythia 410M is a **base causal language model**, not a chat or instruction-tuned assistant. It predicts likely next tokens. It may ignore instructions, repeat itself, invent facts, produce offensive text, or stop unexpectedly. Those failures are part of what this notebook is intended to reveal. Do not use its output as factual advice.

Model card: <https://huggingface.co/EleutherAI/pythia-410m>

## Environment

Before opening this notebook, create the project environment described in `README.md`. Install a CUDA-enabled PyTorch build using the official PyTorch selector, followed by JupyterLab, Transformers, and Safetensors. Restart the kernel after installing or upgrading packages.

In [1]:
from importlib.util import find_spec

required_packages = ["torch", "transformers", "safetensors"]
missing_packages = [name for name in required_packages if find_spec(name) is None]
if missing_packages:
    missing = ", ".join(missing_packages)
    raise RuntimeError(
        f"Missing packages: {missing}. Follow the environment steps in README.md, "
        "then restart this kernel."
    )

import os
import platform
import time
from pathlib import Path

EXPECTED_HF_HOME = Path(r"D:\AI\cache\huggingface").resolve()
EXPECTED_HF_HUB_CACHE = (EXPECTED_HF_HOME / "hub").resolve()
hf_home_value = os.environ.get("HF_HOME")
hf_hub_cache_value = os.environ.get("HF_HUB_CACHE")

print(f"HF_HOME:      {hf_home_value}")
print(f"HF_HUB_CACHE: {hf_hub_cache_value}")

if not hf_home_value or not hf_hub_cache_value:
    raise RuntimeError(
        "HF_HOME and HF_HUB_CACHE must be configured before model loading. "
        "See README.md."
    )

actual_hf_home = Path(hf_home_value).resolve()
actual_hf_hub_cache = Path(hf_hub_cache_value).resolve()
if actual_hf_home != EXPECTED_HF_HOME:
    raise RuntimeError(f"HF_HOME must resolve to {EXPECTED_HF_HOME}, not {actual_hf_home}")
if actual_hf_hub_cache != EXPECTED_HF_HUB_CACHE:
    raise RuntimeError(
        f"HF_HUB_CACHE must resolve to {EXPECTED_HF_HUB_CACHE}, "
        f"not {actual_hf_hub_cache}"
    )
MODEL_CACHE_DIR = EXPECTED_HF_HUB_CACHE

import torch
import transformers
from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    device_name = torch.cuda.get_device_name(0)
else:
    MODEL_DTYPE = torch.float32
    device_name = "CPU"

print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device:       {DEVICE} ({device_name})")
print(f"Model dtype:  {MODEL_DTYPE}")
if DEVICE.type != "cuda":
    print("WARNING: CUDA is not available. The notebook will run on CPU, but generation will be slower.")

HF_HOME:      D:\AI\cache\huggingface
HF_HUB_CACHE: D:\AI\cache\huggingface\hub
Python:       3.11.15
PyTorch:      2.6.0+cu124
Transformers: 5.12.1
Device:       cuda (NVIDIA GeForce RTX 4090)
Model dtype:  torch.bfloat16


## Load the model

The first run downloads the model and tokenizer from Hugging Face. `step143000` is the final training checkpoint and is equivalent to the repository's `main` revision. Keeping it explicit makes the experiment easier to reproduce.

In [2]:
MODEL_ID = "EleutherAI/pythia-410m"
MODEL_REVISION = "step143000"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
)

# Transformers 5 renamed torch_dtype to dtype; support both current APIs.
dtype_argument = (
    {"dtype": MODEL_DTYPE}
    if Version(transformers.__version__) >= Version("5.0.0")
    else {"torch_dtype": MODEL_DTYPE}
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
    use_safetensors=True,
    **dtype_argument,
).to(DEVICE)
model.eval()

if tokenizer.eos_token_id is None:
    raise RuntimeError("The tokenizer has no EOS token; generation cannot be configured safely.")

parameter_count = sum(parameter.numel() for parameter in model.parameters())
resolved_revision = getattr(model.config, "_commit_hash", MODEL_REVISION)
print(f"Loaded:            {MODEL_ID}")
print(f"Requested revision: {MODEL_REVISION}")
print(f"Resolved revision:  {resolved_revision}")
print(f"Parameters:         {parameter_count:,}")
print(f"Context limit:      {model.config.max_position_embeddings:,} tokens")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

C:\Users\arhal\anaconda3\envs\myenv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\AI\cache\huggingface\hub\models--EleutherAI--pythia-410m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded:            EleutherAI/pythia-410m
Requested revision: step143000
Resolved revision:  bba6a464f54bbf08fc174cfb351d9794d58af21d
Parameters:         405,334,016
Context limit:      2,048 tokens


## Generation helper

`generate_continuation` returns only tokens generated after the prompt. Set `temperature=0` for greedy, deterministic decoding. Positive temperatures sample from the model and can expose variability across seeds.

In [3]:
def generate_continuation(
    prompt: str,
    *,
    max_new_tokens: int = 100,
    temperature: float = 0.8,
    top_p: float = 0.95,
    top_k: int = 50,
    seed: int = 42,
) -> dict:
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string")
    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be at least 1")
    if temperature < 0:
        raise ValueError("temperature cannot be negative")
    if not 0 < top_p <= 1:
        raise ValueError("top_p must be in (0, 1]")
    if top_k < 0:
        raise ValueError("top_k cannot be negative")

    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_token_count = encoded["input_ids"].shape[1]
    context_limit = model.config.max_position_embeddings
    if input_token_count + max_new_tokens > context_limit:
        raise ValueError(
            f"Prompt ({input_token_count} tokens) plus requested continuation "
            f"({max_new_tokens} tokens) exceeds the {context_limit}-token context limit."
        )

    encoded = {name: tensor.to(DEVICE) for name, tensor in encoded.items()}
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    do_sample = temperature > 0
    generation_arguments = {
        "max_new_tokens": int(max_new_tokens),
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
    }
    if do_sample:
        generation_arguments.update(
            temperature=float(temperature),
            top_p=float(top_p),
            top_k=int(top_k),
        )

    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**encoded, **generation_arguments)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - started

    new_token_ids = generated[0, input_token_count:]
    continuation = tokenizer.decode(new_token_ids, skip_special_tokens=True)
    new_token_count = int(new_token_ids.numel())
    peak_memory_gb = (
        torch.cuda.max_memory_allocated() / 1024**3
        if DEVICE.type == "cuda"
        else None
    )

    return {
        "prompt": prompt,
        "continuation": continuation,
        "full_text": prompt + continuation,
        "input_tokens": int(input_token_count),
        "new_tokens": new_token_count,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": (new_token_count / elapsed_seconds if elapsed_seconds else None),
        "peak_gpu_memory_gb": peak_memory_gb,
        "seed": seed,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
    }

## Try a prompt

Edit `prompt`, rerun the cell, and compare the continuation with what you expected. Completion-style prompts usually work better than conversational instructions for a base model.

In [4]:
prompt = "Question: What are the limitations of a small language model?\nAnswer:"

In [5]:
result = generate_continuation(
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.95,
    top_k=50,
    seed=42,
)

print("PROMPT")
print(result["prompt"])
print("\nMODEL CONTINUATION")
print(result["continuation"])
print("\nRUN DETAILS")
print(f"Input tokens:      {result['input_tokens']}")
print(f"Generated tokens:  {result['new_tokens']}")
print(f"Elapsed:           {result['elapsed_seconds']:.3f} seconds")
print(f"Generation speed:  {result['tokens_per_second']:.1f} tokens/second")
if result["peak_gpu_memory_gb"] is not None:
    print(f"Peak GPU memory:   {result['peak_gpu_memory_gb']:.2f} GiB")
print(f"Seed:              {result['seed']}")

PROMPT
Question: What are the limitations of a small language model?
Answer:

MODEL CONTINUATION
 A small language model is a model that can only predict two states on a single time instance. It is not a model that can predict a state on multiple time instances. So for example, a model that can only predict two states is a model that can predict state A on multiple instances.

A:

A language model can only predict two states on a single time instance.
It is a very limited model.
It can be a very big language model, which is a very

RUN DETAILS
Input tokens:      15
Generated tokens:  100
Elapsed:           3.703 seconds
Generation speed:  27.0 tokens/second
Peak GPU memory:   0.78 GiB
Seed:              42


## Probe some limitations

The optional cell below compares three prompt forms: factual completion, simple reasoning, and an instruction-like exchange. Look for confident errors, failure to follow the requested format, repetition, sensitivity to wording, and plausible text that does not answer the question.

In [6]:
probe_prompts = {
    "factual completion": "The capital of France is",
    "simple reasoning": (
        "Question: All robins are birds. Some birds cannot fly. "
        "Does it follow that no robins can fly?\nAnswer:"
    ),
    "instruction-like": (
        "User: Explain in exactly one sentence why evidence matters.\nAssistant:"
    ),
}

for label, probe_prompt in probe_prompts.items():
    probe_result = generate_continuation(
        probe_prompt,
        max_new_tokens=60,
        temperature=0.8,
        top_p=0.95,
        top_k=50,
        seed=42,
    )
    print(f"\n--- {label.upper()} ---")
    print(probe_prompt)
    print(probe_result["continuation"])


--- FACTUAL COMPLETION ---
The capital of France is
 in Marseille. The city is located on the coast on the Mediterranean Sea. It is the largest city in France. The city is known for its history and culture. Marseille has a population of 2,721,000. It is located near the French Mediterranean coast.



--- SIMPLE REASONING ---
Question: All robins are birds. Some birds cannot fly. Does it follow that no robins can fly?
Answer:
 A bird cannot fly but a bird can eat. It is not a bird that can fly. It is not a bird that can eat. But, it is a bird that can fly.

We can think about some of the other examples we saw in the previous section and we can also see

--- INSTRUCTION-LIKE ---
User: Explain in exactly one sentence why evidence matters.
Assistant:
 A good idea but I don't think it's very good on a test-
Questioner: In one word
Assistant: Criticism.
Questioner: Why?
Assistant: I think that criticism helps you to judge the evidence.
Questioner: But criticism is not


## What to record

For this first exploration, note:

- whether completion framing performs better than direct instructions;
- whether the output answers the question or merely continues its style;
- factual errors or contradictions;
- repetition and loss of coherence over longer generations;
- changes produced by small prompt edits or different seeds;
- the difference between fluent wording and a justified conclusion.

These observations are behavioural only. They do not yet establish anything about hidden-state geometry or causal mechanisms.